# Stage 2: from-scratch DPO on UltraFeedback preference pairs

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/muttaqi123/small-model-alignment.git"
COMMIT = "5713785"
WORK = Path("/kaggle/working/small-model-alignment")
os.chdir("/kaggle/working")
if not WORK.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(WORK)], check=True)
subprocess.run(["git", "-C", str(WORK), "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", str(WORK), "checkout", "-q", COMMIT], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = str(WORK)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


def run(cmd, env=None):
    """Run a shell command, stream its output, and fail the notebook on error."""
    print(f"$ {cmd}", flush=True)
    merged = {**os.environ, **(env or {})}
    proc = subprocess.Popen(cmd, shell=True, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")


def start(cmd, log_path, env=None):
    """Start a background command (e.g. generation on the second GPU)."""
    print(f"$ {cmd} > {log_path} &", flush=True)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    handle = open(log_path, "w")
    merged = {**os.environ, **(env or {})}
    return subprocess.Popen(cmd, shell=True, env=merged, stdout=handle, stderr=subprocess.STDOUT, text=True), log_path


def wait(job):
    proc, log_path = job
    code = proc.wait()
    print(Path(log_path).read_text()[-3000:])
    if code != 0:
        raise RuntimeError(f"background job failed: {log_path}")


def archive(zip_path, items, exclude=("trainer", "best", "final", "adapter")):
    """Zip small artifacts (metrics, configs, generations, plots); never weights."""
    import zipfile
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in items:
            p = Path(item)
            files = [p] if p.is_file() else sorted(f for f in p.rglob("*") if f.is_file())
            for f in files:
                if any(part in exclude for part in f.parts) or f.suffix in (".safetensors", ".bin", ".pt"):
                    continue
                zf.write(f, str(f))
    print(f"{zip_path}: {Path(zip_path).stat().st_size / 1e6:.1f} MB")


def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"no input matches {pattern}")
    print(f"found {pattern} -> {hits[0]}")
    return hits[0]

In [ ]:
run("nvidia-smi")
run("python -m pip uninstall -y -q torchao || true")
run('python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib')
run("python -m pip install -q -e . --no-deps")
run('python -c "import torch, transformers, datasets; print(\'torch\', torch.__version__, \'| transformers\', transformers.__version__, \'| datasets\', datasets.__version__, \'| cuda\', torch.version.cuda, \'| gpus\', torch.cuda.device_count())"')
run("python -m pytest -q tests/")

In [ ]:
sft_best = str(Path(find_input("outputs/sft/best/config.json")).parent)
data_dir = str(Path(find_input("outputs/data/data_manifest.json")).parent)
Path("outputs/sft").mkdir(parents=True, exist_ok=True)
for f in glob.glob(f"{data_dir}/../sft/*.json*"):
    shutil.copy(f, "outputs/sft/")
print(sft_best, data_dir)

In [ ]:
run(f"python -m alignment.dpo --sft_path {sft_best} --train_path {data_dir}/dpo_uf_train.jsonl "
    f"--val_path {data_dir}/dpo_uf_val.jsonl --output_dir outputs/dpo_uf --eval_every 25")

In [ ]:
run(f"python -m alignment.generate --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --model_path outputs/dpo_uf/best --eval_path eval_sets/alpaca_eval_300_seed42.json --limit 3 "
    "--output_path outputs/generations/sanity_dpo_uf.json --generator qwen2.5-0.5b-dpo-uf")
for r in json.load(open("outputs/generations/sanity_dpo_uf.json")):
    print("=" * 80, "\n", r["instruction"][:300], "\n---\n", r["output"], "\n[finished:", r["finished"], "]")
jobs = [
    start("python -m alignment.generate --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --model_path outputs/dpo_uf/best --eval_path eval_sets/alpaca_eval_300_seed42.json "
          "--output_path outputs/generations/alpaca_dpo_uf.json --generator qwen2.5-0.5b-dpo-uf --dataset alpaca_eval --device cuda:0",
          "outputs/logs/gen_alpaca_dpo_uf.log"),
    start("python -m alignment.generate --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --model_path outputs/dpo_uf/best --eval_path eval_sets/hh_rlhf_eval_300_seed42.json "
          "--output_path outputs/generations/hh_dpo_uf.json --generator qwen2.5-0.5b-dpo-uf --dataset hh_rlhf_style_eval --device cuda:1",
          "outputs/logs/gen_hh_dpo_uf.log"),
]
for job in jobs:
    wait(job)

In [ ]:
run("python scripts/plot_metrics.py --sft outputs/sft --dpo outputs/dpo_uf --out_dir outputs/plots_uf")
archive("dpo_uf_run_outputs.zip", ["outputs/dpo_uf", "outputs/generations", "outputs/logs", "outputs/plots_uf"])
run("du -sh outputs/* && ls -la outputs/dpo_uf")